# 📘 Notebook 10: Word Embeddings

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module D: Meaning as Vectors · Notebook 1 of 2 in this module · (10 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why treating words as isolated symbols is a dead end, and state the **distributional hypothesis**
- Build a **co-occurrence matrix** from a corpus and use its rows as word vectors
- Improve those vectors with **PPMI** and compress them with **SVD**
- Describe the **Word2Vec** skip-gram model with negative sampling, and implement it by hand
- Find **nearest neighbours** and solve **analogies** with pretrained embeddings
- Discuss what embeddings capture, including the **biases** of the text they were learned from

> **Prerequisites:** Notebook 08 (cosine similarity). NumPy arrays and the dot product, from *From Python to Deep Learning & Fine-Tuning*, Notebook 05. For Section 5, the idea of gradient descent from Notebook 09 of that course.
>
> 🔭 **Why this notebook matters:** for three notebooks the same row of our score table has read "no": the machine did not know that two words are similar. This is where that changes. The idea introduced here, that a word can be represented by a short list of numbers learned from the company it keeps, is the foundation of every neural language system built since, including the large language models of today. Their first layer is an embedding table.

> ℹ️ **Setup note.** Run this cell once. It downloads the Brown corpus, about a million words of American English of many kinds.

In [ ]:
import nltk
nltk.download("brown", quiet=True)

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

## 1. Words as isolated symbols

### Intuition first
In every method so far, a word has been a column in a table or a key in a dictionary. *hotel* is column 4,125 and *motel* is column 7,301, and the two numbers have nothing to do with each other.

Written as a vector, such a word is a long row of zeros with a single 1 in its own position. This is called a **one-hot** vector. Any two different one-hot vectors have a dot product of 0, so every word is exactly as different from every other word as it could possibly be.

In [ ]:
vocabulary = ["hotel", "motel", "banana"]

one_hot = np.eye(len(vocabulary))
print(one_hot)
print("hotel . motel  =", one_hot[0] @ one_hot[1])
print("hotel . banana =", one_hot[0] @ one_hot[2])

A system built on these vectors must learn everything about *motel* from scratch, even if it has seen *hotel* a thousand times. We want vectors in which **similar words are close together**.

### The distributional hypothesis
Where could such vectors come from? Try this. You have never seen the word *tezgüino*, but you read:

- *A bottle of tezgüino is on the table.*
- *Everybody likes tezgüino.*
- *Tezgüino makes you drunk.*
- *We make tezgüino out of corn.*

You now have a good idea of what it is: an alcoholic drink, something like beer. You worked it out entirely from the **words around it**, because those are the contexts in which *beer* and *wine* appear too.

> **The distributional hypothesis:** words that occur in similar contexts tend to have similar meanings. In the words of the linguist J. R. Firth (1957): *"You shall know a word by the company it keeps."*

This turns a philosophical question (what is meaning?) into something we can **count**.

## 2. Counting the company a word keeps

### Formal definition
Choose a **window** size, say 2. For every occurrence of a word, its *context words* are the 2 words before it and the 2 words after it. The **co-occurrence matrix** has one row and one column per vocabulary word, and entry $(w, c)$ counts how often $c$ appeared in the window of $w$. Row $w$ of this matrix is a vector describing the company $w$ keeps.

By hand first, on four sentences.

In [ ]:
tiny_corpus = [
    "i drink hot coffee every morning",
    "i drink hot tea every evening",
    "she drinks cold coffee",
    "the car needs new tyres",
]

window = 2
pair_counts = Counter()
for sentence in tiny_corpus:
    words = sentence.split()
    for position, word in enumerate(words):
        start = max(0, position - window)
        end = min(len(words), position + window + 1)
        for neighbour_position in range(start, end):
            if neighbour_position != position:
                pair_counts[(word, words[neighbour_position])] += 1

tiny_vocabulary = sorted(set(word for sentence in tiny_corpus for word in sentence.split()))

def context_vector(word):
    return np.array([pair_counts[(word, context)] for context in tiny_vocabulary])

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("coffee:", dict((c, n) for (w, c), n in pair_counts.items() if w == "coffee"))
print("tea:   ", dict((c, n) for (w, c), n in pair_counts.items() if w == "tea"))
print()
print("coffee and tea:", round(cosine(context_vector("coffee"), context_vector("tea")), 2))
print("coffee and car:", round(cosine(context_vector("coffee"), context_vector("car")), 2))

*coffee* and *tea* never appear in the same sentence. Yet their vectors are similar, because they share neighbours: *drink*, *hot*, *every*. *coffee* and *car* share none. From four sentences, a hint of meaning has already appeared.

### On a million words
Now the Brown corpus. We keep the 5,000 most frequent words as our vocabulary and count co-occurrences with a window of 4. To do this quickly we first replace every word by its number, and then count all pairs at a given distance in one NumPy operation.

In [ ]:
from nltk.corpus import brown

tokens = [word.lower() for word in brown.words() if word.isalpha()]
print(f"{len(tokens):,} tokens")

vocabulary = [word for word, count in Counter(tokens).most_common(5000)]
word_id = {word: i for i, word in enumerate(vocabulary)}
ids = np.array([word_id.get(word, -1) for word in tokens])       # -1 marks a word outside the vocabulary

def cooccurrence_matrix(ids, size, window):
    matrix = np.zeros((size, size), dtype=np.float32)
    for distance in range(1, window + 1):
        left, right = ids[:-distance], ids[distance:]
        both_known = (left >= 0) & (right >= 0)
        np.add.at(matrix, (left[both_known], right[both_known]), 1)     # count each pair ...
        np.add.at(matrix, (right[both_known], left[both_known]), 1)     # ... in both directions
    return matrix

counts = cooccurrence_matrix(ids, len(vocabulary), window=4)
print("Matrix shape:", counts.shape)
print(f"Share of entries that are zero: {(counts == 0).mean():.0%}")

In [ ]:
def nearest(word, vectors, names=None, how_many=6):
    names = names or vocabulary                     # the word that each row of `vectors` belongs to
    target = vectors[names.index(word)]
    similarities = vectors @ target / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(target) + 1e-9)
    best = np.argsort(-similarities)
    return [names[i] for i in best if names[i] != word][:how_many]

for word in ["money", "doctor", "monday", "red", "water"]:
    print(f"{word:<8} {nearest(word, counts)}")

The results are disappointing. The neighbours are dominated by very frequent words, and it is easy to see why. Every word co-occurs a great deal with *the*, *of* and *and*, so those huge counts swamp everything else, and all vectors end up looking alike. We met the same disease in Notebook 08, and the cure is of the same kind.

## 3. Better weights: PPMI

### Intuition first
That *money* appears near *the* 500 times is unremarkable: everything appears near *the*. That *money* appears near *bank* 20 times is remarkable, if the two would be expected to meet only twice by chance. What matters is how much more often two words occur together than they would **if they were unrelated**.

### Formal definition
The **pointwise mutual information** of a word $w$ and a context $c$ compares the observed probability of the pair with the probability expected if they were independent:

$$\text{PMI}(w, c) = \log_2 \frac{P(w, c)}{P(w)\,P(c)}$$

It is positive when the pair occurs more often than chance, and negative when it occurs less often. Negative values are unreliable for rare words, so we replace them with zero. The result is **positive PMI**, or **PPMI**.

In [ ]:
def ppmi(counts):
    total = counts.sum()
    word_probability = counts.sum(axis=1, keepdims=True) / total
    context_probability = counts.sum(axis=0, keepdims=True) / total
    with np.errstate(divide="ignore"):
        pmi = np.log2((counts / total) / (word_probability * context_probability))
    return np.maximum(pmi, 0)

ppmi_vectors = ppmi(counts)

for word in ["money", "doctor", "monday", "red", "water"]:
    print(f"{word:<8} {nearest(word, ppmi_vectors)}")

A change of weighting, with no new data, and something real has appeared: *monday* is now surrounded by other days of the week. Nobody told the program what a day is. It emerged from counting the company each word keeps, as the distributional hypothesis promised.

The other words are still in poor shape, with frequent words crowding their lists. These vectors are 5,000 numbers long, almost all zero, and each one is a noisy estimate from a small corpus. The next step deals with that.

## 4. From sparse to dense: SVD

### Intuition first
Each word is now a vector with 5,000 numbers, most of them zero. That is wasteful, and it has a subtler flaw. The columns *car* and *automobile* are treated as two unrelated contexts, although a word that appears near one would be equally at home near the other.

We would like to **compress** the vectors into a few hundred numbers, in a way that merges columns that behave alike. It is like summarising a person. You could list their answers to 5,000 questions, or you could give a handful of scores (age, interests, temperament) from which most of the answers can be predicted.

### Formal definition
The **singular value decomposition (SVD)** finds the directions along which the rows of a matrix vary the most. Keeping only the top $k$ directions gives each word a **dense** vector of $k$ numbers that preserves as much of the original information as $k$ numbers can. It is the same mathematics as the PCA of *From Python to Deep Learning & Fine-Tuning*, Notebook 11.

A short, dense vector that represents a word is called a **word embedding**.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=0)
embeddings = svd.fit_transform(ppmi_vectors) / svd.singular_values_     # give every direction equal weight

print("Before:", ppmi_vectors.shape, f"{(ppmi_vectors == 0).mean():.0%} zeros")
print("After: ", embeddings.shape)
print()
for word in ["money", "doctor", "monday", "red", "water", "walked"]:
    print(f"{word:<10} {nearest(word, embeddings)}")

Fifty times smaller, and far better. *money* now sits among *taxes*, *income* and *dollars*, and *water* among *salt*, *ice* and *cup*. The compression has done more than save space. By forcing 5,000 noisy numbers through 100, it kept the strong shared patterns and discarded much of the noise.

Look at *walked* too. Several of its neighbours are other verbs of movement in the past tense. The vectors have picked up some **grammar** as well as topic, and again nobody asked them to.

(The division by `svd.singular_values_` is a standard refinement. Without it, the first few directions, which mostly reflect how frequent a word is, would dominate every comparison.)

Let us put a number on the progress, with a few pairs of words.

In [ ]:
pairs = [("car", "automobile"), ("big", "large"), ("doctor", "hospital"), ("car", "music"), ("big", "paper")]

print(f"{'pair':<22} {'one-hot':>8} {'counts':>8} {'PPMI':>8} {'SVD':>8}")
for first, second in pairs:
    i, j = word_id[first], word_id[second]
    row = [0.0, cosine(counts[i], counts[j]), cosine(ppmi_vectors[i], ppmi_vectors[j]), cosine(embeddings[i], embeddings[j])]
    print(f"{first + ' / ' + second:<22}" + "".join(f"{value:>9.2f}" for value in row))

Read the columns from left to right. One-hot vectors say every pair is unrelated. Raw counts say every pair is similar, which is no better. PPMI begins to **separate** the related pairs (the first three rows) from the unrelated ones (the last two), and the dense embeddings separate them clearly. That separation is what a useful representation must provide.

## 5. Learning embeddings by prediction: Word2Vec

### Intuition first
Counting and then compressing works. In 2013 a different route became famous under the name **Word2Vec**. Do not count at all. Start with a random vector for each word, and turn the distributional hypothesis into a guessing game:

> *Given a word, can you tell which words really appear near it from words picked at random?*

Slide a window along the text. Each word and its neighbour form a **true pair**. For each true pair, invent a few **fake pairs** by replacing the neighbour with a random word. Then nudge the vectors so that true pairs get a high score and fake pairs a low one. The score of a pair is simply the dot product of the two vectors.

For *coffee* to score highly with *drink*, *hot* and *cup*, and *tea* to score highly with the same words, the vectors of *coffee* and *tea* are pushed in the same direction, and end up close together. Similarity is never trained directly. It appears as a side effect.

### Formal definition
This is the **skip-gram model with negative sampling**. Each word has two vectors: $\mathbf{v}$, used when it is the centre word, and $\mathbf{u}$, used when it is a context word. For a centre word $w$, a true context $c$ and $K$ random "negative" words $n_1 \dots n_K$, we minimise

$$L = -\log \sigma(\mathbf{u}_c \cdot \mathbf{v}_w) \;-\; \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k} \cdot \mathbf{v}_w)$$

where $\sigma$ is the sigmoid function, which squeezes a score into a probability between 0 and 1. Each term is the loss of a tiny logistic regression: *is this pair real?* The gradient for a single pair has a simple form: **(predicted probability minus true label) times the other vector**.

### By hand, on a toy language
To watch it work we need a corpus small enough to train in seconds, with a structure we can check. So let us make up a language with three kinds of sentence.

In [ ]:
import random

animals = ["cat", "dog", "horse"]
foods = ["bread", "cheese", "meat"]
people = ["king", "queen", "farmer"]
places = ["castle", "village", "city"]

random.seed(0)
toy_sentences = []
for i in range(400):
    kind = random.choice(["eats", "rules", "feeds"])
    if kind == "eats":
        sentence = ["the", random.choice(animals), "eats", "the", random.choice(foods)]
    elif kind == "rules":
        sentence = ["the", random.choice(people), random.choice(["rules", "visits"]), "the", random.choice(places)]
    else:
        sentence = ["the", random.choice(people), "feeds", "the", random.choice(animals)]
    toy_sentences.append(sentence)

for sentence in toy_sentences[:4]:
    print(" ".join(sentence))

toy_vocabulary = sorted(set(word for sentence in toy_sentences for word in sentence))
toy_id = {word: i for i, word in enumerate(toy_vocabulary)}
print(len(toy_vocabulary), "words")

Animals, foods, people and places each appear in their own typical positions. If the method works, it should discover those four groups without being told about them. First we collect the true pairs.

In [ ]:
true_pairs = []
for sentence in toy_sentences:
    for position, word in enumerate(sentence):
        for neighbour_position in range(max(0, position - 2), min(len(sentence), position + 3)):
            if neighbour_position != position:
                true_pairs.append((toy_id[word], toy_id[sentence[neighbour_position]]))

print(len(true_pairs), "true (centre, context) pairs")
print([(toy_vocabulary[a], toy_vocabulary[b]) for a, b in true_pairs[:6]])

Now the training loop. It is the formula above, line for line.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

rng = np.random.default_rng(0)
dimensions = 8
centre_vectors = rng.normal(0, 0.1, (len(toy_vocabulary), dimensions))      # v
context_vectors = rng.normal(0, 0.1, (len(toy_vocabulary), dimensions))     # u
learning_rate = 0.05
negatives = 3

for epoch in range(15):
    total_loss = 0.0
    random.shuffle(true_pairs)
    for centre, context in true_pairs:
        v = centre_vectors[centre]
        # one true pair (label 1) and a few random fake pairs (label 0)
        targets = [context] + list(rng.integers(0, len(toy_vocabulary), negatives))
        labels = [1] + [0] * negatives

        gradient_v = np.zeros(dimensions)
        for target, label in zip(targets, labels):
            u = context_vectors[target]
            predicted = sigmoid(u @ v)
            error = predicted - label                       # prediction minus truth
            total_loss += -np.log(predicted if label == 1 else 1 - predicted)
            gradient_v += error * u
            context_vectors[target] -= learning_rate * error * v
        centre_vectors[centre] -= learning_rate * gradient_v

    if epoch % 2 == 0 or epoch == 14:
        print(f"epoch {epoch:>2}  average loss {total_loss / len(true_pairs):.3f}")

The loss falls, so the model is getting better at telling true pairs from fake ones. That was only ever a pretext. What we care about is the vectors it had to build in order to play the game.

In [ ]:
def toy_nearest(word, how_many=3):
    target = centre_vectors[toy_id[word]]
    similarities = centre_vectors @ target / (np.linalg.norm(centre_vectors, axis=1) * np.linalg.norm(target))
    best = np.argsort(-similarities)
    return [toy_vocabulary[i] for i in best if toy_vocabulary[i] != word][:how_many]

for word in ["cat", "bread", "queen", "village"]:
    print(f"{word:<8} {toy_nearest(word)}")

The vectors live in 8 dimensions. To look at them we project them down to 2 with PCA.

In [ ]:
from sklearn.decomposition import PCA

flat = PCA(n_components=2).fit_transform(centre_vectors)
colours = {"animal": "tab:green", "food": "tab:orange", "person": "tab:blue", "place": "tab:red", "other": "grey"}

plt.figure(figsize=(7, 5))
for word, (x, y) in zip(toy_vocabulary, flat):
    group = ("animal" if word in animals else "food" if word in foods else
             "person" if word in people else "place" if word in places else "other")
    plt.scatter(x, y, color=colours[group])
    plt.annotate(word, (x, y), textcoords="offset points", xytext=(5, 4))
plt.title("Word vectors learned by our skip-gram model (projected to 2 dimensions)")
plt.grid(True)
plt.show()

The words have sorted themselves into their groups. The program was given no labels and no definitions, only sentences and the task of telling real neighbours from random ones. This is **self-supervised learning**: the text supplies its own training signal. The same principle, scaled up by many orders of magnitude, trains the large language models of today.

> 🧠 Counting followed by SVD, and prediction with Word2Vec, look like opposite approaches. They are closely related: it has been shown that skip-gram with negative sampling is implicitly factorising a matrix of PMI values. Two roads, one destination. A third well-known method, **GloVe**, sits explicitly between the two.

## 6. Pretrained embeddings

Our toy model learned from 400 sentences. Embeddings trained on billions of words are freely available, and using them costs one line. The cell below loads **GloVe** vectors (300 numbers per word) through the `sentence-transformers` library, which is already installed in Google Colab, and looks up the vectors of the words in our Brown vocabulary. The first run downloads about 480 MB.

In [ ]:
from sentence_transformers import SentenceTransformer

glove_model = SentenceTransformer("sentence-transformers/average_word_embeddings_glove.6B.300d")

# our 5,000 Brown words, plus a few that the Brown vocabulary lacks and that we want to ask about
extra_words = ["nurse", "tokyo", "rome", "madrid", "spain", "berlin", "programmer", "homemaker", "receptionist"]
glove_words = vocabulary + [word for word in extra_words if word not in word_id]
glove = glove_model.encode(glove_words)         # one row per word

print(glove.shape)
for word in ["money", "doctor", "monday", "red", "water", "walked"]:
    print(f"{word:<10} {nearest(word, glove, glove_words)}")

### Analogies
Good embeddings have a famous property. **Directions** in the space carry meaning. The step from *man* to *woman* is roughly the same as the step from *king* to *queen*. So we can solve *"man is to king as woman is to ...?"* with arithmetic:

$$\text{king} - \text{man} + \text{woman} \approx \;?$$

The function below computes that vector and returns the nearest words. It works with any set of vectors. Here it is applied to the embeddings we built ourselves from the Brown corpus.

In [ ]:
def analogy(a, b, c, vectors, names=None, how_many=3):
    """a is to b as c is to ...?"""
    names = names or vocabulary
    target = vectors[names.index(b)] - vectors[names.index(a)] + vectors[names.index(c)]
    similarities = vectors @ target / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(target) + 1e-9)
    best = np.argsort(-similarities)
    return [names[i] for i in best if names[i] not in (a, b, c)][:how_many]

print("man : king    :: woman : ?   ", analogy("man", "king", "woman", embeddings))
print("walk : walked :: look : ?    ", analogy("walk", "walked", "look", embeddings))
print("big : bigger  :: small : ?   ", analogy("big", "bigger", "small", embeddings))

A million words is very little for this purpose, and our home-made vectors solve analogies unreliably. With the pretrained vectors the picture changes.

In [ ]:
print("man : king    :: woman : ?   ", analogy("man", "king", "woman", glove, glove_words))
print("walk : walked :: look : ?    ", analogy("walk", "walked", "look", glove, glove_words))
print("big : bigger  :: small : ?   ", analogy("big", "bigger", "small", glove, glove_words))
print("france : paris :: italy : ?  ", analogy("france", "paris", "italy", glove, glove_words))

### Embeddings inherit the views of their text
Embeddings are learned from what people have written, and they absorb its regularities without discrimination: the useful ones (*Paris* is to *France* as *Rome* is to *Italy*) and the regrettable ones.

In [ ]:
print("man : doctor     :: woman : ?   ", analogy("man", "doctor", "woman", glove, glove_words))
print("man : programmer :: woman : ?   ", analogy("man", "programmer", "woman", glove, glove_words))
print("woman : nurse    :: man : ?     ", analogy("woman", "nurse", "man", glove, glove_words))

Look at what comes back and ask whether you would want a system that screens job applications or ranks search results to reason in this way. Embeddings trained on ordinary text have repeatedly been shown to encode stereotypes about gender, ethnicity and age, because the text does. The model is not malicious. It is a mirror of its data.

> ⚠️ **Common pitfalls**
>
> - Treating an embedding as a definition. It records **usage** in one body of text. Train on medical journals and *cell* sits near *membrane*. Train on news and it sits near *prison* and *phone*.
> - Forgetting that each word gets exactly **one** vector. *bank* has a single point for the river and the money, somewhere between the two. This limitation was only overcome by the contextual models of Module F.
> - Assuming that similar means synonymous. *hot* and *cold* are close neighbours, because they appear in the same contexts. Embeddings capture **relatedness**, which is not the same thing as sameness of meaning.
> - Building embeddings into a decision system without checking for bias.

## 7. Where we stand

| | Bag of words + TF-IDF | Word embeddings |
|---|---|---|
| Turns text into numbers | yes, one sparse vector per document | yes, one dense vector per **word** |
| Knows that two words are similar | no | **yes** |
| Size of a vector | the whole vocabulary | a few hundred numbers |
| Uses word order | no | no |
| One word, several meanings | no | no |

The row that read "no" for three notebooks has finally changed. Two remain. A sentence is more than a set of word vectors: order matters, and Module E brings in neural networks that read words in sequence. And a word's meaning depends on the sentence it is in, which is the problem solved by the contextual models of Module F.

---
## ✏️ Exercises

### Exercise 1 (Co-occurrence by hand)
For the corpus `["the cat sat", "the dog sat", "the cat ran"]` and a window of **1** (only the immediate neighbours), write down on paper the context counts of `cat` and of `dog`. Then compute their cosine similarity in code. Why is it high although the two words never occur together?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
mini_corpus = ["the cat sat", "the dog sat", "the cat ran"]
mini_counts = Counter()
for sentence in mini_corpus:
    words = sentence.split()
    for position, word in enumerate(words):
        for neighbour_position in [position - 1, position + 1]:
            if 0 <= neighbour_position < len(words):
                mini_counts[(word, words[neighbour_position])] += 1

mini_vocabulary = ["cat", "dog", "ran", "sat", "the"]
cat = np.array([mini_counts[("cat", context)] for context in mini_vocabulary])
dog = np.array([mini_counts[("dog", context)] for context in mini_vocabulary])

print(dict(zip(mini_vocabulary, cat)))
print(dict(zip(mini_vocabulary, dog)))
print(round(cosine(cat, dog), 2))
```

*cat: the 2, sat 1, ran 1. dog: the 1, sat 1. They share the neighbours "the" and "sat". Similar words rarely sit next to each other. They sit in the same places.*
</details>

### Exercise 2 (Explore the neighbourhoods)
Print the six nearest neighbours of `king`, `water`, `ran`, `children` and `quickly` using the 100-dimensional `embeddings`. For which words are the neighbours related by **topic**, and for which by **grammatical form**?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for word in ["king", "water", "ran", "children", "quickly"]:
    print(f"{word:<10} {nearest(word, embeddings)}")
```

*Nouns such as "water" and "children" attract words on the same topic. A past-tense verb such as "ran" attracts other past-tense verbs of movement. A word's contexts reflect both what it means and how it is used grammatically, and the vectors store both. The neighbours of "quickly" are a mixture, a reminder that a million words is a small corpus for this purpose.*
</details>

### Exercise 3 (The size of the window)
Build two new sets of embeddings from the Brown corpus, one with `window=1` and one with `window=10` (co-occurrence counts, then PPMI, then SVD to 100 dimensions). Compare the nearest neighbours of `hospital`, `walked` and `good`. What changes?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def build_embeddings(window):
    matrix = ppmi(cooccurrence_matrix(ids, len(vocabulary), window))
    model = TruncatedSVD(n_components=100, random_state=0)
    return model.fit_transform(matrix) / model.singular_values_

narrow = build_embeddings(1)
wide = build_embeddings(10)

for word in ["hospital", "walked", "good"]:
    print(word)
    print("   window 1: ", nearest(word, narrow))
    print("   window 10:", nearest(word, wide))
```

*A narrow window sees only the immediate neighbours, so it groups words that can replace one another in a sentence: words of the same grammatical type. A wide window sees the whole surrounding passage, so it groups words that belong to the same topic. The window size decides what kind of similarity the embeddings express.*
</details>

### Exercise 4 (The odd one out)
Write `odd_one_out(words, vectors)` that returns the word least similar to the others: for each word, compute its average cosine similarity to the rest, and return the word with the lowest average. Test it on `["monday", "tuesday", "friday", "river"]`, `["red", "blue", "green", "idea"]` and `["doctor", "medical", "hospital", "music"]`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def odd_one_out(words, vectors):
    averages = {}
    for word in words:
        total = 0.0
        for other in words:
            if other != word:
                total += cosine(vectors[word_id[word]], vectors[word_id[other]])
        averages[word] = total / (len(words) - 1)
    return min(averages, key=averages.get)

for group in [["monday", "tuesday", "friday", "river"], ["red", "blue", "green", "idea"], ["doctor", "medical", "hospital", "music"]]:
    print(group, "->", odd_one_out(group, embeddings))
```

*A task that sounds as if it needs understanding has become a few lines of geometry. Once words are points in a space, many questions about meaning turn into questions about distance.*
</details>

### Exercise 5 (Experiment with the toy Word2Vec)
Copy the training loop of Section 5 into a function `train_toy(dimensions, negatives, epochs)` that returns the trained centre vectors and the final average loss. Run it with 2 dimensions and with 8, each for 15 epochs with 3 negatives. For each, print the nearest neighbour of `cat` and of `king`. Are two dimensions enough for this language?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def train_toy(dimensions, negatives, epochs):
    rng = np.random.default_rng(0)
    centre = rng.normal(0, 0.1, (len(toy_vocabulary), dimensions))
    context = rng.normal(0, 0.1, (len(toy_vocabulary), dimensions))
    pairs = list(true_pairs)
    for epoch in range(epochs):
        total_loss = 0.0
        random.shuffle(pairs)
        for centre_word, context_word in pairs:
            v = centre[centre_word]
            targets = [context_word] + list(rng.integers(0, len(toy_vocabulary), negatives))
            labels = [1] + [0] * negatives
            gradient_v = np.zeros(dimensions)
            for target, label in zip(targets, labels):
                u = context[target]
                predicted = sigmoid(u @ v)
                total_loss += -np.log(predicted if label == 1 else 1 - predicted)
                gradient_v += (predicted - label) * u
                context[target] -= 0.05 * (predicted - label) * v
            centre[centre_word] -= 0.05 * gradient_v
    return centre, total_loss / len(pairs)

for dimensions in [2, 8]:
    vectors, loss = train_toy(dimensions, 3, 15)
    result = {}
    for word in ["cat", "king"]:
        target = vectors[toy_id[word]]
        similarities = vectors @ target / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(target))
        result[word] = [toy_vocabulary[i] for i in np.argsort(-similarities) if toy_vocabulary[i] != word][0]
    print(f"{dimensions} dimensions: loss {loss:.3f}  nearest {result}")
```

*With only two numbers per word the model has too little room: its loss stays higher and the neighbours come out wrong. Eight dimensions are enough for this small language. Real vocabularies have vastly more structure to store, which is why practical embeddings use 100 to 1,000 dimensions.*
</details>

### Exercise 6 (Analogies with pretrained vectors)
This exercise needs the `glove` vectors of Section 6, so run it in Colab. Try the analogies below and two of your own. Which kinds of relation do the vectors capture well?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
questions = [
    ("good", "better", "bad"),
    ("france", "french", "germany"),
    ("boy", "girl", "brother"),
    ("day", "days", "year"),
    ("japan", "tokyo", "england"),
]
for a, b, c in questions:
    print(f"{a} : {b} :: {c} : ?   {analogy(a, b, c, glove, glove_words)}")
```

*Regular grammatical relations (plural, comparative, nationality) and well-documented facts such as capital cities are captured strikingly well. Relations that are rare in text, or that depend on a particular sense of a word, are much less reliable.*
</details>

## 🔑 Key takeaways

- **One-hot** vectors make every word equally unrelated to every other. We need vectors in which similar words are close.
- The **distributional hypothesis**: words that occur in similar contexts have similar meanings. Meaning becomes something we can count.
- A **co-occurrence matrix** gives each word a vector of its contexts. **PPMI** reweights it by how surprising each co-occurrence is, and **SVD** compresses it into a dense **embedding**.
- **Word2Vec** learns embeddings by prediction: tell true neighbours from random words. Similarity emerges as a side effect. This is **self-supervised** learning.
- In a good embedding space, **neighbours** are related words and **directions** carry meaning, which makes analogies solvable by arithmetic.
- Embeddings record usage in their training text, **including its biases**, and give each word a single vector whatever its sense.

---
**Next:** *Notebook 11: Topic Modelling*, where the same idea, discovering hidden structure from co-occurrence, is applied to whole documents: what themes run through a collection that nobody has labelled?

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*